In [1]:
# Step 1: Import libraries and load the API key
import os
import requests
import pandas as pd
from dotenv import load_dotenv

load_dotenv()   # reads the .env file
api_key = os.getenv("TMDB_API_KEY")

print("Key loaded:", api_key is not None)

Key loaded: True


In [2]:
# Step 2: Test the API with one movie (Avengers: Endgame)
url = "https://api.themoviedb.org/3/movie/299534"
response = requests.get(url, params={"api_key": api_key})

print("Status code:", response.status_code)
print("Title:", response.json().get("title"))

Status code: 200
Title: Avengers: Endgame


In [3]:
# Step 3: Fetch all movies from the lab list
movie_ids = [0, 299534, 19995, 140607, 299536, 597, 135397,
             420818, 24428, 168259, 99861, 284054, 12445,
             181808, 330457, 351286, 109445, 321612, 260513]

movies = []   # empty list to collect each movie

for movie_id in movie_ids:
    url = "https://api.themoviedb.org/3/movie/" + str(movie_id)
    params = {"api_key": api_key, "append_to_response": "credits"}
    response = requests.get(url, params=params)

    if response.status_code == 200:
        movies.append(response.json())
        print("OK:", movie_id)
    else:
        print("Not found:", movie_id)

df = pd.DataFrame(movies)
print("Number of movies:", len(df))

Not found: 0
OK: 299534
OK: 19995
OK: 140607
OK: 299536
OK: 597
OK: 135397
OK: 420818
OK: 24428
OK: 168259
OK: 99861
OK: 284054
OK: 12445
OK: 181808
OK: 330457
OK: 351286
OK: 109445
OK: 321612
OK: 260513
Number of movies: 18


In [4]:
# Step 4: Drop irrelevant columns
columns_to_drop = ["adult", "imdb_id", "original_title", "video", "homepage"]
df = df.drop(columns=columns_to_drop)

print("Columns left:", len(df.columns))
print(df.columns.tolist())

Columns left: 23
['backdrop_path', 'belongs_to_collection', 'budget', 'genres', 'id', 'origin_country', 'original_language', 'overview', 'popularity', 'poster_path', 'production_companies', 'production_countries', 'release_date', 'revenue', 'runtime', 'softcore', 'spoken_languages', 'status', 'tagline', 'title', 'vote_average', 'vote_count', 'credits']


In [5]:
# Step 5: Extract names from JSON-like columns

def get_names(value, key="name"):
    # value is a list of dictionaries -> join the names with "|"
    if type(value) != list:
        return None
    names = []
    for item in value:
        names.append(item[key])
    return "|".join(names)

def get_collection(value):
    # value is one dictionary, or empty if the movie has no collection
    if type(value) == dict:
        return value["name"]
    return None

df["genres"] = df["genres"].apply(get_names)
df["production_countries"] = df["production_countries"].apply(get_names)
df["production_companies"] = df["production_companies"].apply(get_names)
df["spoken_languages"] = df["spoken_languages"].apply(get_names, key="english_name")
df["belongs_to_collection"] = df["belongs_to_collection"].apply(get_collection)

df[["title", "genres", "belongs_to_collection", "spoken_languages"]].head()

,title,genres,belongs_to_collection,spoken_languages
0,Avengers: Endgame,Adventure|Science Fiction|Action,The Avengers Collection,English|Japanese|Xhosa
1,Avatar,Science Fiction|Action|Adventure,Avatar Collection,English|Spanish
2,Star Wars: The Force Awakens,Adventure|Action|Science Fiction,Star Wars Collection,English
3,Avengers: Infinity War,Adventure|Action|Science Fiction,The Avengers Collection,English|Xhosa
4,Titanic,Drama|Romance,None,English|French|German|Swedish|Italian|Russian


In [6]:
# Step 6: Extract cast and director from credits

def get_cast(credits):
    # names of all actors, joined with "|"
    names = []
    for person in credits["cast"]:
        names.append(person["name"])
    return "|".join(names)

def get_director(credits):
    # find the crew member whose job is "Director"
    for person in credits["crew"]:
        if person["job"] == "Director":
            return person["name"]
    return None

df["cast"] = df["credits"].apply(get_cast)
df["cast_size"] = df["credits"].apply(lambda c: len(c["cast"]))
df["director"] = df["credits"].apply(get_director)
df["crew_size"] = df["credits"].apply(lambda c: len(c["crew"]))

df = df.drop(columns=["credits"])

df[["title", "director", "cast_size", "crew_size"]].head()

,title,director,cast_size,crew_size
0,Avengers: Endgame,Anthony Russo,103,613
1,Avatar,James Cameron,68,999
2,Star Wars: The Force Awakens,J.J. Abrams,99,270
3,Avengers: Infinity War,Joe Russo,69,737
4,Titanic,James Cameron,116,271


In [7]:
# Step 7: Fix data types and convert money to millions

# Make sure these are numbers (bad values become NaN)
df["budget"] = pd.to_numeric(df["budget"], errors="coerce")
df["revenue"] = pd.to_numeric(df["revenue"], errors="coerce")
df["id"] = pd.to_numeric(df["id"], errors="coerce")
df["popularity"] = pd.to_numeric(df["popularity"], errors="coerce")
df["release_date"] = pd.to_datetime(df["release_date"], errors="coerce")

# 0 is not a real budget, revenue or runtime -> treat it as missing
for column in ["budget", "revenue", "runtime"]:
    df[column] = df[column].replace(0, float("nan"))

# Convert to million USD and drop the old columns
df["budget_musd"] = df["budget"] / 1000000
df["revenue_musd"] = df["revenue"] / 1000000
df = df.drop(columns=["budget", "revenue"])

df[["title", "release_date", "budget_musd", "revenue_musd", "runtime"]].head()

,title,release_date,budget_musd,revenue_musd,runtime
0,Avengers: Endgame,2019-04-24,356.0,2927.449455,181
1,Avatar,2009-12-16,237.0,2923.706026,162
2,Star Wars: The Force Awakens,2015-12-15,245.0,2068.223624,136
3,Avengers: Infinity War,2018-04-25,300.0,2052.415039,149
4,Titanic,1997-12-18,200.0,2264.162353,194


In [8]:
# Step 8: Finish cleaning (lab points 6 to 9)

# Movies with 0 votes: their rating means nothing -> set it to missing
df.loc[df["vote_count"] == 0, "vote_average"] = float("nan")

# Placeholder text in overview and tagline -> missing
for column in ["overview", "tagline"]:
    df[column] = df[column].replace(["No Data", ""], float("nan"))

print("Rows at start:", len(df))

# Remove duplicate movies and rows with no id or title
df = df.drop_duplicates(subset="id")
df = df.dropna(subset=["id", "title"])
print("After duplicates and missing id/title:", len(df))

# Keep only rows with at least 10 filled-in columns
df = df.dropna(thresh=10)
print("After keeping rows with 10+ values:", len(df))

# Keep only released movies, then drop the status column
df = df[df["status"] == "Released"]
df = df.drop(columns=["status"])
print("After keeping released movies:", len(df))

Rows at start: 18
After duplicates and missing id/title: 18
After keeping rows with 10+ values: 18
After keeping released movies: 18


In [9]:
# Step 9: Reorder columns and reset the index (lab points 10 and 11)
column_order = ["id", "title", "tagline", "release_date", "genres", "belongs_to_collection",
                "original_language", "budget_musd", "revenue_musd", "production_companies",
                "production_countries", "vote_count", "vote_average", "popularity", "runtime",
                "overview", "spoken_languages", "poster_path", "cast", "cast_size", "director", "crew_size"]

df = df[column_order]
df = df.reset_index(drop=True)

print("Final shape (rows, columns):", df.shape)
df.head()

Final shape (rows, columns): (18, 22)


,id,title,tagline,release_date,genres,belongs_to_collection,original_language,budget_musd,revenue_musd,production_companies,...,vote_average,popularity,runtime,overview,spoken_languages,poster_path,cast,cast_size,director,crew_size
0,299534,Avengers: Endgame,Avenge the fallen.,2019-04-24,Adventure|Science Fiction|Action,The Avengers Collection,en,356.0,2927.449455,Marvel Studios,...,8.200,96.8577,181,After the devastating events of Avengers: Infi...,English|Japanese|Xhosa,/bR8ISy1O9XQxqiy0fQFw2BX72RQ.jpg,Robert Downey Jr.|Chris Evans|Mark Ruffalo|Chr...,103,Anthony Russo,613
1,19995,Avatar,Enter the world of Pandora.,2009-12-16,Science Fiction|Action|Adventure,Avatar Collection,en,237.0,2923.706026,Dune Entertainment|Lightstorm Entertainment|20...,...,7.600,41.8923,162,"In the 22nd century, a paraplegic Marine is di...",English|Spanish,/gKY6q7SjCkAU6FqvqWybDYgUKIF.jpg,Sam Worthington|Zoe Saldaña|Sigourney Weaver|S...,68,James Cameron,999
2,140607,Star Wars: The Force Awakens,Every generation has a story.,2015-12-15,Adventure|Action|Science Fiction,Star Wars Collection,en,245.0,2068.223624,Lucasfilm Ltd.|Bad Robot,...,7.200,25.1323,136,Thirty years after defeating the Galactic Empi...,English,/wqnLdwVXoBjKibFRR5U3y0aDUhs.jpg,Daisy Ridley|John Boyega|Oscar Isaac|Adam Driv...,99,J.J. Abrams,270
3,299536,Avengers: Infinity War,An entire universe. Once and for all.,2018-04-25,Adventure|Action|Science Fiction,The Avengers Collection,en,300.0,2052.415039,Marvel Studios,...,8.245,93.1399,149,As the Avengers and their allies have continue...,English|Xhosa,/7WsyChQLEftFiDOVTGkv3hFpyyt.jpg,Robert Downey Jr.|Chris Evans|Chris Hemsworth|...,69,Joe Russo,737
4,597,Titanic,Nothing on earth could come between them.,1997-12-18,Drama|Romance,None,en,200.0,2264.162353,Paramount Pictures|20th Century Fox|Lightstorm...,...,7.900,39.3321,194,101-year-old Rose DeWitt Bukater tells the sto...,English|French|German|Swedish|Italian|Russian,/9xjZS2rlVxm8SFx8kPC3aIGCOYQ.jpg,Leonardo DiCaprio|Kate Winslet|Billy Zane|Kath...,116,James Cameron,271
